# 01 — Preprocessing delle corse TLC

## 1. Obiettivo e dataset

Prepariamo le corse registrate a New York nel luglio 2025 per il confronto RDD e Spark SQL del notebook 02. I tre file originali in `data/raw/` sono in formato Parquet, un formato a colonne che Spark può leggere direttamente.

| Categoria | Corse registrate | File originale |
|---|---|---|
| Yellow | Taxi gialli | `yellow_tripdata_2025-07.parquet` |
| Green | Taxi verdi | `green_tripdata_2025-07.parquet` |
| HVFHV | Veicoli a noleggio con conducente di operatori ad alto volume | `fhvhv_tripdata_2025-07.parquet` |

Il notebook uniforma le colonne delle tre fonti, filtra le corse e salva un Parquet pulito per categoria in HDFS. Il report `results/quality.json` conserva soglie, schema di origine e conteggi.

## 2. Configurazione e inizializzazione di Spark

Controlliamo la presenza dei tre input e avviamo Spark con la configurazione del progetto. La tabella mostra le impostazioni effettive della sessione. I timestamp rappresentano l'orario civile di New York e sono trattati come `TIMESTAMP_NTZ`. La sessione usa UTC per mantenere i valori di calendario durante la conversione del tipo.

In [1]:
from IPython.display import Markdown, display

from src.config import CLEANED, SLOTS, THRESHOLDS, raw_paths, spark_session
from src.preprocessing import preprocess

nomi_categorie = {"yellow": "Yellow", "green": "Green", "hvfhv": "HVFHV"}
verifica_conteggi_hdfs = True


def mostra_tabella(intestazioni, righe):
    tabella = [
        "| " + " | ".join(intestazioni) + " |",
        "| " + " | ".join("---" for _ in intestazioni) + " |",
    ]
    tabella.extend("| " + " | ".join(map(str, riga)) + " |" for riga in righe)
    display(Markdown("\n".join(tabella)))


def intero(valore):
    return f"{valore:,}".replace(",", ".")


paths = raw_paths()
spark = spark_session("TLC 01 - Preprocessing")

mostra_tabella(("Impostazione", "Valore"), [
    ("Versione Spark", spark.version),
    ("Master", spark.sparkContext.master),
    ("Memoria del driver", spark.sparkContext.getConf().get("spark.driver.memory", "predefinita")),
    ("Partizioni di shuffle", spark.conf.get("spark.sql.shuffle.partitions")),
    ("Fuso della sessione", spark.conf.get("spark.sql.session.timeZone")),
    ("Radice output", CLEANED),
])

26/11/20 22:26:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


| Impostazione | Valore |
| --- | --- |
| Versione Spark | 3.5.5 |
| Master | local[4] |
| Memoria del driver | 3g |
| Partizioni di shuffle | 12 |
| Fuso della sessione | UTC |
| Radice output | hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned |

## 3. Regole di qualità e soglie adottate

Una corsa è conservata se supera tutti i controlli. Le soglie di seguito provengono dalla configurazione usata per filtrare i dati. Le statistiche per regola possono contare la stessa corsa più volte. Il totale degli scarti conta ogni corsa esclusa una volta sola.

La durata è la differenza tra l'ora di arrivo e l'ora di partenza. La velocità si ricava dalla distanza e dalla durata della corsa. Il ricavo è la tariffa base (`fare_amount` per Yellow e Green, `base_passenger_fare` per HVFHV). Giorno della settimana e fascia oraria dipendono dal pickup. Il giorno vale 1 per lunedì e 7 per domenica.

In [2]:
mostra_tabella(("Controllo", f"Criterio di ammissione (v{THRESHOLDS['version']})"), [
    ("Ora di partenza (pickup)",
     f"Da {THRESHOLDS['pickup_start']} incluso a {THRESHOLDS['pickup_end']} escluso"),
    ("Timestamp", "Partenza e arrivo entrambi presenti"),
    ("Durata", f"Maggiore di 0 e al massimo {intero(THRESHOLDS['max_duration_seconds'])} s"),
    ("Distanza", f"Finita, maggiore di 0 e al massimo {THRESHOLDS['max_distance_miles']} mi"),
    ("Velocità", f"Finita, maggiore di 0 e al massimo {THRESHOLDS['max_speed_mph']} mph"),
    ("Ricavo", f"Finito e almeno {THRESHOLDS['min_revenue_usd']} USD"),
])
display(Markdown("Fasce orarie di quattro ore: " + ", ".join(SLOTS)))

| Controllo | Criterio di ammissione (v1) |
| --- | --- |
| Ora di partenza (pickup) | Da 2025-07-01 00:00:00 incluso a 2025-08-01 00:00:00 escluso |
| Timestamp | Partenza e arrivo entrambi presenti |
| Durata | Maggiore di 0 e al massimo 14.400 s |
| Distanza | Finita, maggiore di 0 e al massimo 200 mi |
| Velocità | Finita, maggiore di 0 e al massimo 100 mph |
| Ricavo | Finito e almeno 0 USD |

Fasce orarie di quattro ore: 00-04, 04-08, 08-12, 12-16, 16-20, 20-24

## 4. Esecuzione del preprocessing

Per ogni categoria Spark apre il Parquet originale, calcola le statistiche di qualità e scrive le corse valide in HDFS. I messaggi indicano la fase raggiunta. Dopo la scrittura si prova a leggere una riga per verificare che il Parquet sia accessibile e che contenga dati quando sono attese corse valide.

In [3]:
report = preprocess(spark, paths)

[22:27:19 UTC] Avvio preprocessing 20261120T222719Z-80fd1769. Spark 3.5.5, master local[4], memoria driver 3g, partizioni shuffle 12, fuso sessione UTC.
[22:27:19 UTC] Soglie v1. Pickup in [2025-07-01 00:00:00, 2025-08-01 00:00:00), durata fino a 14400 s, distanza fino a 200 mi, velocità fino a 100 mph, ricavo minimo 0 USD.
[22:27:19 UTC] Yellow — inizio lettura del Parquet originale: yellow_tripdata_2025-07.parquet
[22:27:21 UTC] Yellow — inizio calcolo delle statistiche e controlli di qualità


[Stage 1:==============>                                            (1 + 3) / 4]

[22:27:25 UTC] Yellow — 3898963 record originali, 405122 esclusi e 3493841 validi.
[22:27:25 UTC] Yellow — inizio scrittura su HDFS: hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/yellow


[Stage 4:=============================>                             (2 + 2) / 4]

[22:27:37 UTC] Yellow — scrittura su HDFS completata.
[22:27:37 UTC] Yellow — verifica di leggibilità su HDFS


[22:27:37 UTC] Yellow — verifica di leggibilità su HDFS completata.
[22:27:37 UTC] Green — inizio lettura del Parquet originale: green_tripdata_2025-07.parquet
[22:27:37 UTC] Green — inizio calcolo delle statistiche e controlli di qualità
[22:27:38 UTC] Green — 48205 record originali, 2008 esclusi e 46197 validi.
[22:27:38 UTC] Green — inizio scrittura su HDFS: hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/green
[22:27:38 UTC] Green — scrittura su HDFS completata.
[22:27:38 UTC] Green — verifica di leggibilità su HDFS
[22:27:39 UTC] Green — verifica di leggibilità su HDFS completata.
[22:27:39 UTC] HVFHV — inizio lettura del Parquet originale: fhvhv_tripdata_2025-07.parquet
[22:27:39 UTC] HVFHV — inizio calcolo delle statistiche e controlli di qualità


[Stage 15:==============>                                           (1 + 3) / 4]

[22:27:42 UTC] HVFHV — 19653012 record originali, 3028 esclusi e 19649984 validi.
[22:27:42 UTC] HVFHV — inizio scrittura su HDFS: hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/hvfhv


[Stage 18:=============================>                            (2 + 2) / 4]

[22:28:18 UTC] HVFHV — scrittura su HDFS completata.
[22:28:18 UTC] HVFHV — verifica di leggibilità su HDFS


[22:28:19 UTC] HVFHV — verifica di leggibilità su HDFS completata.
[22:28:19 UTC] Salvataggio del report: /workspace/results/quality.json
[22:28:19 UTC] Preprocessing completato per 3 categorie. 23600180 record originali, 410158 esclusi e 23190022 validi.


## 5. Riepilogo dei record conservati ed esclusi

La tabella affianca i conteggi di input e output ai metadati essenziali dei tre file. La percentuale di scarto usa gli esclusi unici come numeratore.

In [4]:
categorie = report["categories"]
riepilogo = []
for categoria, dati in categorie.items():
    conteggi = dati["counts"]
    riepilogo.append((
        nomi_categorie[categoria],
        intero(len(dati["schema"])),
        f"{dati['size_bytes'] / 2**20:.2f}".replace(".", ","),
        intero(conteggi["original_count"]),
        intero(conteggi["removed_unique"]),
        intero(conteggi["valid_count"]),
        f"{dati['removed_pct']:.2f}".replace(".", ","),
    ))

mostra_tabella(
    ("Categoria", "Colonne originali", "MiB originali", "Record originali",
     "Esclusi unici", "Conservati", "Esclusi (%)"),
    riepilogo,
)

| Categoria | Colonne originali | MiB originali | Record originali | Esclusi unici | Conservati | Esclusi (%) |
| --- | --- | --- | --- | --- | --- | --- |
| Yellow | 20 | 63,84 | 3.898.963 | 405.122 | 3.493.841 | 10,39 |
| Green | 21 | 1,13 | 48.205 | 2.008 | 46.197 | 4,17 |
| HVFHV | 25 | 464,65 | 19.653.012 | 3.028 | 19.649.984 | 0,02 |

## 6. Dettaglio delle anomalie per categoria

Ogni riga mostra quante corse violano uno specifico controllo. I conteggi delle righe si sovrappongono e quindi la loro somma può superare il numero di corse escluse.

La seconda tabella riporta le verifiche diagnostiche. Il trattino indica che il dato non è previsto per quella categoria.

In [5]:
regole = {
    "pickup_outside_month": "Pickup fuori dall'intervallo",
    "null_timestamp": "Timestamp mancanti",
    "duration_nonpositive": "Durata non positiva",
    "duration_over_limit": "Durata oltre soglia",
    "distance_invalid": "Distanza mancante, non finita o non positiva",
    "distance_over_limit": "Distanza oltre soglia",
    "speed_nonpositive_or_non_finite": "Velocità calcolabile non positiva o non finita",
    "speed_over_limit": "Velocità oltre soglia",
    "revenue_invalid": "Ricavo mancante, non finito o sotto soglia",
}
intestazioni_categorie = tuple(nomi_categorie[categoria] for categoria in categorie)
mostra_tabella(
    ("Controllo violato", *intestazioni_categorie),
    [(nome, *(intero(dati["counts"][chiave]) for dati in categorie.values()))
     for chiave, nome in regole.items()],
)

diagnostica = {
    "valid_revenue_subcent": "Ricavi validi con frazioni di centesimo",
    "negative_fare_physically_valid": "Tariffa negativa con gli altri controlli superati",
    "negative_fare_physically_valid_positive_total": "Stesso caso con importo totale positivo",
    "trip_time_diff_over_60s": "Scarto oltre 60 s fra durata calcolata e dichiarata",
}
mostra_tabella(
    ("Indicatore diagnostico", *intestazioni_categorie),
    [(nome, *(intero(dati["counts"][chiave]) if chiave in dati["counts"] else "—"
              for dati in categorie.values()))
     for chiave, nome in diagnostica.items()],
)

| Controllo violato | Yellow | Green | HVFHV |
| --- | --- | --- | --- |
| Pickup fuori dall'intervallo | 7 | 13 | 0 |
| Timestamp mancanti | 0 | 0 | 0 |
| Durata non positiva | 56.064 | 40 | 1 |
| Durata oltre soglia | 1.073 | 215 | 373 |
| Distanza mancante, non finita o non positiva | 123.774 | 1.516 | 2.155 |
| Distanza oltre soglia | 201 | 14 | 155 |
| Velocità calcolabile non positiva o non finita | 0 | 0 | 0 |
| Velocità oltre soglia | 947 | 205 | 4 |
| Ricavo mancante, non finito o sotto soglia | 247.088 | 130 | 470 |

| Indicatore diagnostico | Yellow | Green | HVFHV |
| --- | --- | --- | --- |
| Ricavi validi con frazioni di centesimo | 0 | 0 | 0 |
| Tariffa negativa con gli altri controlli superati | 224.798 | — | — |
| Stesso caso con importo totale positivo | 153.517 | — | — |
| Scarto oltre 60 s fra durata calcolata e dichiarata | — | — | 724 |

## 7. Verifica dei file Parquet salvati su HDFS

Questa cella rilegge i Parquet puliti e conta esattamente le righe con una sola chiamata a `count()` per categoria. Confronta i conteggi con i record validi del report e segnala eventuali differenze. I conteggi completi sono una verifica separata dalla prova di leggibilità del preprocessing. Per saltarli in una riesecuzione, impostare `verifica_conteggi_hdfs = False` nella sezione 2.

In [6]:
if verifica_conteggi_hdfs:
    verifiche = []
    for categoria, dati in categorie.items():
        nome = nomi_categorie[categoria]
        percorso = dati["cleaned"]
        attesi = dati["counts"]["valid_count"]
        print(f"{nome} — conteggio delle righe rilette da {percorso}", flush=True)
        riletti = spark.read.parquet(percorso).count()
        verifiche.append((nome, percorso, attesi, riletti))

    mostra_tabella(
        ("Categoria", "Percorso HDFS", "Record attesi", "Record riletti", "Esito"),
        [(nome, percorso, intero(attesi), intero(riletti), "OK" if attesi == riletti else "DIFF")
         for nome, percorso, attesi, riletti in verifiche],
    )
    difformi = [nome for nome, _, attesi, riletti in verifiche if attesi != riletti]
    if difformi:
        raise RuntimeError(f"Conteggi Parquet diversi da quelli attesi per {', '.join(difformi)}")
else:
    display(Markdown("Conteggio completo dei Parquet saltato per configurazione."))

Yellow — conteggio delle righe rilette da hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/yellow
Green — conteggio delle righe rilette da hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/green
HVFHV — conteggio delle righe rilette da hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/hvfhv


| Categoria | Percorso HDFS | Record attesi | Record riletti | Esito |
| --- | --- | --- | --- | --- |
| Yellow | hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/yellow | 3.493.841 | 3.493.841 | OK |
| Green | hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/green | 46.197 | 46.197 | OK |
| HVFHV | hdfs://hdfs:9000/data/nyc-tlc/2025-07/cleaned/hvfhv | 19.649.984 | 19.649.984 | OK |

## 8. Chiusura della sessione Spark

Una volta completate le verifiche, chiudiamo la sessione per liberare le risorse di Spark.

In [7]:
spark.stop()